In [1]:
import geopandas as gpd
import pandas as pd
import shapely
import rasterio
from pathlib import Path
from rasterstats import zonal_stats
from exactextract import exact_extract

In [2]:

points = gpd.read_parquet("./data/clean.parquet")[["geo_lat", "geo_lon", "geometry"]]
points = points.drop_duplicates(subset=["geo_lat", "geo_lon"])
points = points.to_crs(32637)
points = points.reset_index(drop=True)
points.shape

(46639, 3)

In [3]:
def calc_raster_stats(raster_path, points, buffer_size=500, stats=["mean"]):
    buffered_gdf = points.copy()
    buffered_gdf["geometry"] = points.buffer(buffer_size)

    with rasterio.open(raster_path) as src:
        assert src.crs == points.crs

        result_stats = exact_extract(
            src, buffered_gdf, stats, 
            output="pandas")

    return result_stats.add_suffix(f"_{buffer_size}")

    

In [4]:
rasters_path = Path("./data/raster")
ndvi_filename = "ndvi_2021.tif"
ndbi_filename = "ndbi_2021.tif"

In [5]:
rasters = {"ndvi": rasters_path / ndvi_filename, "ndbi": rasters_path / ndbi_filename}

parts = []
for name, path in rasters.items():
    for radius in [300, 500, 1000]:
        stats = calc_raster_stats(path, points, radius, ["mean", "stdev"])
        parts.append(stats.add_prefix(f"{name}_"))



In [10]:
features = pd.concat([points.drop(columns="geometry").reset_index(drop=True)] + parts, axis=1)
assert features.shape[0] == features[["geo_lat", "geo_lon"]].drop_duplicates().shape[0]

Path("./data/geo/").mkdir(exist_ok=True)
features.to_parquet("./data/geo/raster_stats.parquet", index=False)

In [11]:
features.head()

,geo_lat,geo_lon,ndvi_mean_300,ndvi_stdev_300,ndvi_mean_500,ndvi_stdev_500,ndvi_mean_1000,ndvi_stdev_1000,ndbi_mean_300,ndbi_stdev_300,ndbi_mean_500,ndbi_stdev_500,ndbi_mean_1000,ndbi_stdev_1000
0,55.738876,37.825370,0.620381,0.185425,0.624330,0.193340,0.578413,0.242396,-0.244237,0.132130,-0.242016,0.132131,-0.210943,0.160993
1,55.798921,37.738090,0.587791,0.215085,0.523747,0.257394,0.448090,0.263030,-0.196036,0.132806,-0.162893,0.155660,-0.114029,0.161708
2,55.655307,37.614605,0.541558,0.225406,0.504596,0.250352,0.461879,0.250968,-0.169720,0.145262,-0.152236,0.156566,-0.124095,0.155246
3,55.701330,37.507412,0.366494,0.203973,0.390460,0.206833,0.405170,0.236107,-0.064431,0.121425,-0.075794,0.122668,-0.082716,0.139498
4,55.730785,37.631424,0.216355,0.206954,0.232510,0.213303,0.247250,0.209198,0.022269,0.138934,0.011841,0.138641,-0.001401,0.129547


In [9]:
parts

[       ndvi_mean_300  ndvi_stdev_300
 0           0.620381        0.185425
 1           0.587791        0.215085
 2           0.541558        0.225406
 3           0.366494        0.203973
 4           0.216355        0.206954
 ...              ...             ...
 46634       0.587731        0.212243
 46635       0.172587        0.171912
 46636       0.238821        0.207519
 46637       0.320743        0.235451
 46638       0.412559        0.249043
 
 [46639 rows x 2 columns],
        ndvi_mean_500  ndvi_stdev_500
 0           0.624330        0.193340
 1           0.523747        0.257394
 2           0.504596        0.250352
 3           0.390460        0.206833
 4           0.232510        0.213303
 ...              ...             ...
 46634       0.573343        0.227962
 46635       0.296273        0.243378
 46636       0.271431        0.209498
 46637       0.368130        0.231315
 46638       0.410724        0.257013
 
 [46639 rows x 2 columns],
        ndvi_mean_1000  ndvi_s